# 20. 데이터 사전 분할 (Asymmetric Underbagging Subsets)

이 노트북은 4,700만 건의 거대한 학습 데이터를 **원하는 설정**에 따라 여러 개의 서브셋으로 분할하여 저장합니다.
여기서 생성된 데이터는 `data/train_subsets/seed_{SEED}/` 경로에 저장되며, 학습 시 자동으로 인식됩니다.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from pathlib import Path
from src.train_core import AsymmetricSampler
import config.train_config as cfg

# ── [독립 설정 구역] ────────────────────────────────────────
# 학습 컨피그와 별개로, 여기서 원하는 분할 조건을 자유롭게 설정하세요.

TARGET_SEED = 42      # 생성할 데이터셋의 시드 번호
N_SUBSETS   = 10      # 생성할 서브셋 개수
NEG_RATIO   = 10      # 정상:고장 비율 (기본 10:1)
NEAR_WINDOW = 30      # Near-failure 구간 (D-day ~ D-10(10일): failure, D-11 ~ D-30(20일): near-failure, 고장일은 이미 삭제했으니 실제 계산 시 하루씩 당김)
NEAR_WEIGHT = 3.0     # Near-failure 가중치

print(f"🚀 독립 분할 작업 준비: Seed={TARGET_SEED}, Subsets={N_SUBSETS}, Ratio={NEG_RATIO}:1")

🖥️  LightGBM GPU 가용성: [OK] 사용 가능
🚀 독립 분할 작업 준비: Seed=42, Subsets=10, Ratio=10:1


## 1. 데이터 로드
원본 데이터(`train.parquet`)를 로드합니다.

In [2]:
print(f"📂 원본 데이터 로딩 중... ({cfg.TRAIN_PATH})")
df_train = pd.read_parquet(cfg.TRAIN_PATH)
print(f"✅ 로드 완료: {len(df_train):,} rows")

📂 원본 데이터 로딩 중... (c:\Workspace\06_ML_projdect\26_1_COIN\data\split_group_stratified\train.parquet)
✅ 로드 완료: 47,252,074 rows


## 2. 데이터 분할 실행

In [3]:
print("🔧 Asymmetric Underbagging 분할 중...")
sampler = AsymmetricSampler(
    n_subsets=N_SUBSETS,
    neg_ratio=NEG_RATIO,
    near_window=NEAR_WINDOW,
    near_weight=NEAR_WEIGHT,
    seed=TARGET_SEED
)

subsets = sampler.split(df_train, target_col=cfg.TARGET_COL)
print(f"\n✅ {len(subsets)}개의 서브셋 생성 완료")

🔧 Asymmetric Underbagging 분할 중...
✅  [Sampler] 10개 서브셋 생성 완료
   pos=33,437  neg/subset=334,370  total/subset=367,807

✅ 10개의 서브셋 생성 완료


## 3. 파일 저장
`data/train_subsets/seed_{TARGET_SEED}/` 폴더에 저장합니다.

In [4]:
# 상대 경로를 절대 경로로 변환하여 안전하게 저장
save_dir = (Path("..") / "data" / "train_subsets" / f"seed_{TARGET_SEED}").resolve()
save_dir.mkdir(parents=True, exist_ok=True)

print(f"💾 저장 경로: {save_dir}")

for i, sub in enumerate(subsets):
    path = save_dir / f"subset_{i}.parquet"
    sub.to_parquet(path, index=False)
    print(f"  [Done] Subset {i+1}/{len(subsets)}: {len(sub):,} rows")

print(f"\n✨ Seed {TARGET_SEED} 버전의 데이터셋 생성이 완료되었습니다!")

💾 저장 경로: C:\Workspace\06_ML_projdect\26_1_COIN\data\train_subsets\seed_42
  [Done] Subset 1/10: 367,807 rows
  [Done] Subset 2/10: 367,807 rows
  [Done] Subset 3/10: 367,807 rows
  [Done] Subset 4/10: 367,807 rows
  [Done] Subset 5/10: 367,807 rows
  [Done] Subset 6/10: 367,807 rows
  [Done] Subset 7/10: 367,807 rows
  [Done] Subset 8/10: 367,807 rows
  [Done] Subset 9/10: 367,807 rows
  [Done] Subset 10/10: 367,807 rows

✨ Seed 42 버전의 데이터셋 생성이 완료되었습니다!
